# LeNet-5 Model Experiments

In [1]:
import torch
import numpy as np
import torchvision
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor
from torch.utils.data.sampler import SubsetRandomSampler
from torch.utils.data.dataloader import DataLoader
import matplotlib.pyplot as plt
%matplotlib inline

## Setting up datasets for training, validation, and testing

In [2]:
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.Pad(2),   # 28x28 -> 32x32
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)) # Standard MNIST mean/std
])

dataset = MNIST(root='data/', download=True, transform=transform)
len(dataset)

60000

In [3]:
test_dataset = MNIST(root='data/', train=False, transform=transform)
len(test_dataset)

10000

In [4]:
# global random seed for all experiments to ensure reproducibility
seed = 42
torch.manual_seed(seed)
g = torch.Generator()
g.manual_seed(seed)

In [5]:
from torch.utils.data import random_split

val_size = 10000
train_size = len(dataset) - val_size

train_ds, val_ds = random_split(dataset, [train_size, val_size], generator=g)
len(train_ds), len(val_ds)

(50000, 10000)

### Dataloaders Set

In [6]:
batch_size = 128

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    generator=g,
    pin_memory=False
)

val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False,
    pin_memory=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

## Experiments

In [7]:
# get the model
import sys
import os

# Adds the parent directory to the python path
sys.path.append(os.path.abspath('..'))
from models.lenet5 import LeNet5, fit, evaluate, mle_loss

In [8]:
import torch.nn as nn

def init_weights(m):
    # 'm' is each layer (module) in your network
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
        # Apply Xavier/Glorot initialization
        torch.nn.init.xavier_uniform_(m.weight)
        # Set biases to zero to start neutral
        if m.bias is not None:
            nn.init.zeros_(m.bias)

### Baseline Model (Controlled Experiment)

In [9]:
from time import time
torch.manual_seed(seed)
baseline_model = LeNet5(
    activation="tanh",
    classifier="rbf",
    pooling="avg",
    batchnorm=False
)
baseline_model.apply(init_weights)

# parameters
epochs = 10
lr = 0.01

# training
start_time = time() # start the timer
baseline_history = fit(
    epochs=epochs,
    lr=lr, 
    model=baseline_model,
    train_loader=train_loader, 
    val_loader=val_loader,
    opt_func=torch.optim.SGD
)
elapsed = time() - start_time    
print(f"Time elapsed {elapsed}")

Epoch [0], val_loss: 0.5648, val_acc: 0.8536
Epoch [1], val_loss: 0.8878, val_acc: 0.7940
Epoch [2], val_loss: 0.2240, val_acc: 0.9395
Epoch [3], val_loss: 0.2089, val_acc: 0.9417
Epoch [4], val_loss: 0.1885, val_acc: 0.9502
Epoch [5], val_loss: 0.1734, val_acc: 0.9512
Epoch [6], val_loss: 0.1683, val_acc: 0.9538
Epoch [7], val_loss: 0.1645, val_acc: 0.9560
Epoch [8], val_loss: 0.1612, val_acc: 0.9575
Epoch [9], val_loss: 0.1592, val_acc: 0.9561
Time elapsed 216.80531358718872


In [10]:
# test
baseline_result = evaluate(baseline_model, test_loader)
baseline_result

{'val_loss': 0.14884047210216522, 'val_acc': 0.957772970199585}

In [11]:
# time in minutes and seconds
def recorded_time(elapsed_time):
    minutes = int(elapsed_time // 60)
    seconds = round((elapsed_time / 60 - minutes) * 60)
    print(f"time: {minutes}m {seconds}s")

recorded_time(elapsed)

time: 3m 37s


### RELU Experiment

In [12]:
from models.lenet5 import LeNet5, fit, evaluate
torch.manual_seed(seed)
relu_model = LeNet5(
    activation="relu",
    classifier="rbf",
    pooling="avg",
    batchnorm=False
)

# parameters
epochs = 10
lr = 0.01

# training
start_time = time() # start the timer
relu_history = fit(
    epochs=epochs,
    lr=lr, 
    model=relu_model,
    train_loader=train_loader, 
    val_loader=val_loader,
    opt_func=torch.optim.SGD
)
elapsed = time() - start_time    
recorded_time(elapsed)

Epoch [0], val_loss: 1.2070, val_acc: 0.5746
Epoch [1], val_loss: 1.1174, val_acc: 0.6398
Epoch [2], val_loss: 1.0716, val_acc: 0.6838
Epoch [3], val_loss: 1.0534, val_acc: 0.7024
Epoch [4], val_loss: 1.0433, val_acc: 0.6985
Epoch [5], val_loss: 1.0350, val_acc: 0.7155
Epoch [6], val_loss: 1.0312, val_acc: 0.7113
Epoch [7], val_loss: 1.0286, val_acc: 0.7110
Epoch [8], val_loss: 1.0266, val_acc: 0.7135
Epoch [9], val_loss: 1.0243, val_acc: 0.7175
time: 3m 22s


In [13]:
# test
relu_result = evaluate(relu_model, test_loader)
relu_result

{'val_loss': 1.0114606618881226, 'val_acc': 0.7212222814559937}

### ADAM Experiement

In [14]:
from models.lenet5 import LeNet5, fit, evaluate
torch.manual_seed(seed)
adam_model = LeNet5(
    activation="tanh",
    classifier="rbf",
    pooling="avg",
    batchnorm=False
)
adam_model.apply(init_weights)

# parameters
epochs = 10
lr = 0.01

# training
start_time = time() # start the timer
adam_history = fit(
    epochs=epochs,
    lr=lr, 
    model=adam_model,
    train_loader=train_loader, 
    val_loader=val_loader,
    opt_func=torch.optim.Adam
)
elapsed = time() - start_time    
recorded_time(elapsed)

Epoch [0], val_loss: 0.3723, val_acc: 0.9081
Epoch [1], val_loss: 0.4445, val_acc: 0.8993
Epoch [2], val_loss: 0.2471, val_acc: 0.9397
Epoch [3], val_loss: 0.2644, val_acc: 0.9392
Epoch [4], val_loss: 0.1725, val_acc: 0.9557
Epoch [5], val_loss: 0.2036, val_acc: 0.9514
Epoch [6], val_loss: 0.1644, val_acc: 0.9605
Epoch [7], val_loss: 0.1724, val_acc: 0.9589
Epoch [8], val_loss: 0.1498, val_acc: 0.9617
Epoch [9], val_loss: 0.1523, val_acc: 0.9627
time: 3m 55s


In [15]:
# test
adam_result = evaluate(adam_model, test_loader)
adam_result

{'val_loss': 0.13455523550510406, 'val_acc': 0.9642009735107422}

### Maxpooling Experiment

In [16]:
from models.lenet5 import LeNet5, fit, evaluate
torch.manual_seed(seed)
max_model = LeNet5(
    activation="tanh",
    classifier="rbf",
    pooling="max",
    batchnorm=False
)
max_model.apply(init_weights)

# parameters
epochs = 10
lr = 0.01

# training
start_time = time() # start the timer
max_history = fit(
    epochs=epochs,
    lr=lr, 
    model=max_model,
    train_loader=train_loader, 
    val_loader=val_loader,
    opt_func=torch.optim.SGD
)
elapsed = time() - start_time    
recorded_time(elapsed)

Epoch [0], val_loss: 0.6092, val_acc: 0.8290
Epoch [1], val_loss: 0.3758, val_acc: 0.8968
Epoch [2], val_loss: 0.3006, val_acc: 0.9178
Epoch [3], val_loss: 0.2748, val_acc: 0.9220
Epoch [4], val_loss: 0.2603, val_acc: 0.9271
Epoch [5], val_loss: 0.2440, val_acc: 0.9325
Epoch [6], val_loss: 0.2288, val_acc: 0.9374
Epoch [7], val_loss: 0.2232, val_acc: 0.9380
Epoch [8], val_loss: 0.2195, val_acc: 0.9413
Epoch [9], val_loss: 0.2173, val_acc: 0.9413
time: 3m 58s


In [17]:
# test
max_result = evaluate(max_model, test_loader)
max_result

{'val_loss': 0.21213485300540924, 'val_acc': 0.9400712251663208}

### Learnable Classifier Experiment

Utilizes softmax for classifying and cross entropy for its loss function.

In [18]:
from models.lenet5 import LeNet5, fit, evaluate
torch.manual_seed(seed)
learnable_model = LeNet5(
    activation="tanh",
    classifier="linear",
    pooling="avg",
    batchnorm=False
)
learnable_model.apply(init_weights)

# parameters
epochs = 10
lr = 0.01

# training
start_time = time() # start the timer
learnable_history = fit(
    epochs=epochs,
    lr=lr, 
    model=learnable_model,
    train_loader=train_loader, 
    val_loader=val_loader,
    opt_func=torch.optim.SGD
)
elapsed = time() - start_time    
recorded_time(elapsed)

Epoch [0], val_loss: 0.3772, val_acc: 0.8966
Epoch [1], val_loss: 0.3083, val_acc: 0.9100
Epoch [2], val_loss: 0.2813, val_acc: 0.9176
Epoch [3], val_loss: 0.2595, val_acc: 0.9246
Epoch [4], val_loss: 0.2506, val_acc: 0.9260
Epoch [5], val_loss: 0.2420, val_acc: 0.9276
Epoch [6], val_loss: 0.2367, val_acc: 0.9312
Epoch [7], val_loss: 0.2328, val_acc: 0.9316
Epoch [8], val_loss: 0.2304, val_acc: 0.9322
Epoch [9], val_loss: 0.2282, val_acc: 0.9328
time: 4m 14s


In [19]:
# test
learnable_result = evaluate(learnable_model, test_loader)
learnable_result

{'val_loss': 0.20904789865016937, 'val_acc': 0.9381922483444214}

### Batchnorm Experiment

In [20]:
from models.lenet5 import LeNet5, fit, evaluate
torch.manual_seed(seed)
batchnorm_model = LeNet5(
    activation="tanh",
    classifier="rbf",
    pooling="avg",
    batchnorm=True
)
batchnorm_model.apply(init_weights)

# parameters
epochs = 10
lr = 0.01

# training
start_time = time() # start the timer
batchnorm_history = fit(
    epochs=epochs,
    lr=lr, 
    model=batchnorm_model,
    train_loader=train_loader, 
    val_loader=val_loader
)
elapsed = time() - start_time    
recorded_time(elapsed)

Epoch [0], val_loss: 0.2977, val_acc: 0.9212
Epoch [1], val_loss: 0.2397, val_acc: 0.9331
Epoch [2], val_loss: 0.1739, val_acc: 0.9499
Epoch [3], val_loss: 0.1693, val_acc: 0.9517
Epoch [4], val_loss: 0.1516, val_acc: 0.9590
Epoch [5], val_loss: 0.1459, val_acc: 0.9600
Epoch [6], val_loss: 0.1453, val_acc: 0.9600
Epoch [7], val_loss: 0.1408, val_acc: 0.9608
Epoch [8], val_loss: 0.1392, val_acc: 0.9623
Epoch [9], val_loss: 0.1387, val_acc: 0.9623
time: 4m 31s


In [21]:
# test
batchnorm_result = evaluate(batchnorm_model, test_loader)
batchnorm_result

{'val_loss': 0.12060786783695221, 'val_acc': 0.964102029800415}

### All Modern Improvements Experiment

In [22]:
from models.lenet5 import LeNet5, fit, evaluate
torch.manual_seed(seed)
modern_model = LeNet5(
    activation="relu",
    classifier="linear",
    pooling="max",
    batchnorm=True
)

# parameters
epochs = 10
lr = 0.01

# training
start_time = time() # start the timer
modern_history = fit(
    epochs=epochs,
    lr=lr, 
    model=modern_model,
    train_loader=train_loader, 
    val_loader=val_loader,
    opt_func=torch.optim.Adam
)
elapsed = time() - start_time    
recorded_time(elapsed)

Epoch [0], val_loss: 0.0766, val_acc: 0.9749
Epoch [1], val_loss: 0.0599, val_acc: 0.9831
Epoch [2], val_loss: 0.0480, val_acc: 0.9869
Epoch [3], val_loss: 0.0555, val_acc: 0.9856
Epoch [4], val_loss: 0.0403, val_acc: 0.9897
Epoch [5], val_loss: 0.0474, val_acc: 0.9876
Epoch [6], val_loss: 0.0409, val_acc: 0.9903
Epoch [7], val_loss: 0.0429, val_acc: 0.9906
Epoch [8], val_loss: 0.0431, val_acc: 0.9902
Epoch [9], val_loss: 0.0465, val_acc: 0.9903
time: 4m 4s


In [23]:
# test
modern_result = evaluate(modern_model, test_loader)
modern_result

{'val_loss': 0.03032461367547512, 'val_acc': 0.9926819801330566}

### RELU + Learnable Classifier

In [24]:
from models.lenet5 import LeNet5, fit, evaluate
torch.manual_seed(seed)
relu_v2_model = LeNet5(
    activation="relu",
    classifier="linear",
    pooling="avg",
    batchnorm=False
)

# parameters
epochs = 10
lr = 0.01

# training
start_time = time() # start the timer
relu_v2_history = fit(
    epochs=epochs,
    lr=lr, 
    model=relu_v2_model,
    train_loader=train_loader, 
    val_loader=val_loader,
    opt_func=torch.optim.SGD
)
elapsed = time() - start_time    
recorded_time(elapsed)

Epoch [0], val_loss: 1.6151, val_acc: 0.6417
Epoch [1], val_loss: 0.4681, val_acc: 0.8555
Epoch [2], val_loss: 0.3793, val_acc: 0.8876
Epoch [3], val_loss: 0.3404, val_acc: 0.9011
Epoch [4], val_loss: 0.3183, val_acc: 0.9062
Epoch [5], val_loss: 0.3030, val_acc: 0.9085
Epoch [6], val_loss: 0.2959, val_acc: 0.9120
Epoch [7], val_loss: 0.2888, val_acc: 0.9122
Epoch [8], val_loss: 0.2856, val_acc: 0.9130
Epoch [9], val_loss: 0.2824, val_acc: 0.9134
time: 3m 45s


In [25]:
# test
relu_v2_result = evaluate(relu_v2_model, test_loader)
relu_v2_result

{'val_loss': 0.2604103982448578, 'val_acc': 0.9231606125831604}

| Model                           | Activation  | Pooling | Optimizer | Classifier       | BatchNorm | Training Time | Test Accuracy | Notes                                         |
| ------------------------------- | ----------- | ------- | --------- | ---------------- | --------- | ------------- | ------------- | --------------------------------------------- |
| **Baseline LeNet-5**            | Scaled Tanh | AvgPool | SGD       | RBF              | No        | 3m 37s        | **95.78%**    | Reproduction of original LeNet-5 architecture |
| **ReLU**                        | ReLU        | AvgPool | SGD       | RBF              | No        | 3m 22s        | 72.12%        | Activation mismatch with RBF classifier       |
| **MaxPool**                     | Scaled Tanh | MaxPool | SGD       | RBF              | No        | 3m 58s        | 94.01%        | Replace AvgPool with MaxPool                  |
| **Adam**                        | Scaled Tanh | AvgPool | Adam      | RBF              | No        | 3m 55s        | 96.42%        | Replace SGD with Adam optimizer               |
| **Softmax Classifier**          | Scaled Tanh | AvgPool | SGD       | Linear + Softmax | No        | 4m 14s        | 93.82%        | Learnable fully connected classifier          |
| **BatchNorm**                   | Scaled Tanh | AvgPool | SGD       | RBF              | Yes       | 4m 31s        | 96.41%        | Batch normalization after convolution layers  |
| **ReLU + Learnable Classifier** | ReLU        | AvgPool | SGD       | Linear + Softmax | No        | 3m 45s        | 92.32%        | ReLU paired with Softmax classifier           |
| **Modern LeNet**                | ReLU        | MaxPool | Adam      | Linear + Softmax | Yes       | 4m 04s        | **99.27%**    | Combined modern improvements                  |
